In [ ]:
import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
tfd = tfp.distributions
tfb = tfp.bijectors
vi = tfp.vi

import matplotlib.pyplot as plt
from modulars.tfp_rr_test import make_conditioned_lp
from tqdm.notebook import tqdm
from modulars.plot_rr import plot_a_few_trajectories_1d, plot_mean_band_rrs_1d
from modulars import gaussian_1d_posterior, gaussian_1d
from modulars.tfp_rr_test import tfp_run_restart_1d

In [2]:
mu_prior, sigma_prior = 0, 10
sigma_like = 1.0
max_iters = 100_000



DATA = True
if DATA:
    data = gaussian_1d(100, 3, sigma_like, seed=0).astype(np.float32)
    true_mu_post, true_sigma_post = gaussian_1d_posterior(data, mu_prior=mu_prior, sigma_prior=sigma_prior, sigma_like=sigma_like)

else:
    data = []
    true_mu_post, true_sigma_post = mu_prior, sigma_prior


conditioned_log_prob = make_conditioned_lp(
  prior_dist = tfd.Normal(mu_prior, sigma_prior),
  likelihood_dist = lambda z: tfd.Normal(z, sigma_like),
  x = data
)


In [3]:
results = []
for seed in tqdm(range(100)):
    result = tfp_run_restart_1d(
        seed, conditioned_log_prob)
    results.append(result)

  0%|          | 0/100 [00:00<?, ?it/s]

2026-03-19 13:03:25.305368: W tensorflow/compiler/tf2xla/kernels/random_ops.cc:108] Warning: Using tf.random.uniform with XLA compilation will ignore seeds; consider using tf.random.stateless_uniform instead if reproducible behavior is desired. fit_surrogate_posterior/sanitize_seed/seed
I0000 00:00:1773939805.583912 4356494 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [ ]:
%matplotlib inline
from modulars import apply_traj_transform, save_rr_tracking_csv
TRACKING_CSV = "processed_tracking/rr_tfp_tracking.csv"

single_means, single_stds, multi_means, multi_stds = \
apply_traj_transform(results, transform_fn=None,
                     n_samples=10_000, seed=1, NOTEBOOK=True)
save_rr_tracking_csv(
    TRACKING_CSV,
    {"default": (single_means, single_stds, multi_means, multi_stds)},
)


In [ ]:
%matplotlib inline
from modulars import load_rr_tracking_csv

TRACKING_CSV = "processed_tracking/rr_tfp_tracking.csv"

single_means, single_stds, multi_means, multi_stds, x = load_rr_tracking_csv(
    TRACKING_CSV, scenario="default"
)
N, T = single_means.shape

# If we have "best" from config, these should be on theta-scale (0,1)
best_mu, best_std = true_mu_post, true_sigma_post


plot_a_few_trajectories_1d(
    [single_means, single_stds], [multi_means, multi_stds],
    best_mu, best_std, r'$\sigma^2$')
plot_mean_band_rrs_1d(
    single_means, single_stds, best_mu, best_std,
    x, r'$\sigma^2$', 1)
plot_mean_band_rrs_1d(
    multi_means, multi_stds, best_mu, best_std,
    x, r'$\sigma^2$', 100)
